# Diseño de la red de suministro

## Tipo de problema

Estructurar los flujos globales de bienes respondiendo a aranceles, fluctuaciones de divisas, tiempos de tránsito y costes operativos.

## Formas de resolverlo

Los nombres de la primera columna coinciden con la [tabla del README](../README.md#formas-de-resolverlo).

| Enfoque (README) | Aplicación en este problema | Uso en este ejercicio |
|---|---|---|
| Entera o mixta | Combinar aperturas, capacidades y flujos con datos conocidos. | Alternativa |
| Modelos estocásticos y simulación | Decidir aperturas antes de observar la demanda y adaptar flujos por escenario; aquí no se realiza simulación. | **Principal** |
| Enumeración | Comparar aperturas comunes y envíos factibles para cada escenario pequeño. | **Complementario** |

**Enfoque utilizado:** Modelos estocásticos y simulación.

**Enfoque complementario:** Enumeración.

**Técnica concreta:** Modelo de dos etapas con recurso, resuelto enumerando aperturas y envíos factibles por escenario.

## Ejercicio propuesto

Elegir centros antes de conocer la demanda y decidir envíos después.  
Centros Norte y Sur: apertura 5 y 9 €, capacidad 4 y 6 ud.  
Costes €/ud a clientes C1,C2: Norte=(1,4), Sur=(3,1).  
Escenario bajo (prob. 0.6): demanda (2,2); alto (prob. 0.4): (3,3).  
Toda demanda debe cubrirse en ambos escenarios. Origen con oferta ilimitada;  
los costes unitarios incluyen suministro y entrega. Apertura pagada una vez.  
Objetivo: apertura + transporte esperado. Método exacto de dos etapas:  
enumerar aperturas comunes y optimizar flujos enteros por escenario (recurso).  
No se elige una apertura distinta después de observar cada escenario.

## Método elegido para este ejercicio

**Enfoque utilizado:** Modelos estocásticos y simulación.

**Enfoque complementario:** Enumeración.

Optimización estocástica de dos etapas con recurso: se enumeran aperturas comunes y se optimizan los envíos por escenario. El ejemplo simplifica los factores globales a costes unitarios y demanda incierta; no modela aranceles ni divisas por separado.

## Implementación

Ejecuta las celdas de código en orden. Solo se utiliza la biblioteca estándar de Python.

In [1]:
from itertools import product


def resolver():
    escenarios = (('Bajo', 0.6, (2, 2)), ('Alto', 0.4, (3, 3)))
    mejor = float('inf'), None, None
    for abiertos in product((0, 1), repeat=2):
        esperado = 5 * abiertos[0] + 9 * abiertos[1]
        politica = []
        for nombre, probabilidad, demanda in escenarios:
            minimo, envios = float('inf'), None
            for norte in product(*(range(d + 1) for d in demanda)):
                sur = tuple(d - n for d, n in zip(demanda, norte))
                if sum(norte) > 4 * abiertos[0] or sum(sur) > 6 * abiertos[1]:
                    continue
                coste = norte[0] + 4 * norte[1] + 3 * sur[0] + sur[1]
                if coste < minimo:
                    minimo, envios = coste, (norte, sur)
            if envios is None:
                break
            esperado += probabilidad * minimo
            politica.append((nombre, envios, minimo))
        else:
            if esperado < mejor[0]:
                mejor = esperado, abiertos, politica
    assert mejor[1] is not None
    return mejor

## Resolución y resultados

In [2]:
coste, abiertos, politica = resolver()
print('Abrir: ' + ', '.join(n for n, a in zip(('Norte', 'Sur'), abiertos) if a))
for escenario, (norte, sur), transporte in politica:
    print(f'{escenario}: Norte->(C1,C2)={norte}; Sur->(C1,C2)={sur}; coste={transporte} €')
print(f'Coste esperado mínimo: {coste:.2f} €')

Abrir: Sur
Bajo: Norte->(C1,C2)=(0, 0); Sur->(C1,C2)=(2, 2); coste=8 €
Alto: Norte->(C1,C2)=(0, 0); Sur->(C1,C2)=(3, 3); coste=12 €
Coste esperado mínimo: 18.60 €


## Interpretación

Abrir solo **Sur**, que puede cubrir ambos escenarios. El coste mínimo esperado es **18,60 €**: 9 de apertura + 0,6 × 8 + 0,4 × 12 de transporte. Los envíos se adaptan a la demanda; la apertura se decide antes.

**Alcance:** los datos son sintéticos. Las garantías y supuestos se refieren a esta instancia; al aumentar su tamaño, la enumeración puede resultar costosa.